# Raw API Data Ingestion — Environmental Data Lakehouse

This notebook performs **raw ingestion** of environmental data from three APIs:
1. **Open-Meteo Weather API** — comprehensive current, hourly, and daily forecast variables
2. **Open-Meteo Air Quality API** — pollutants, pollen, UV, and AQI indices
3. **USGS Earthquake API** — complete GeoJSON feed (all_day summary)

Raw JSON responses are saved to `/Volumes/weather_api/raw/raw_files/` in source-specific subdirectories.
Each extraction produces a uniquely named, timestamped file. No transformations are applied.

**Naming convention:**
- `weather_<Location>_<YYYYMMDDTHHMMSSZ>.json`
- `air_quality_<Location>_<YYYYMMDDTHHMMSSZ>.json`
- `earthquakes_<YYYYMMDDTHHMMSSZ>.json`

In [0]:
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import yaml
from dotenv import load_dotenv

# ── Project paths ──
PROJECT_ROOT = Path("/Workspace/Users/adityarajsharma3030@gmail.com/weather-data-pipeline")
LOCATIONS_FILE = PROJECT_ROOT / "config" / "locations.yaml"

# ── Import API client modules from the Git folder ──
sys.path.insert(0, str(PROJECT_ROOT / "src" / "ingestion"))
import weather_api
import air_quality_api
import usgs_api

# Load API URLs from .env (with fallback to module defaults)
load_dotenv(PROJECT_ROOT / ".env")
WEATHER_API_URL = os.getenv("WEATHER_API_URL", weather_api.DEFAULT_API_URL)
AIR_QUALITY_API_URL = os.getenv("AIR_QUALITY_API_URL", air_quality_api.DEFAULT_API_URL)
USGS_API_URL = os.getenv("USGS_API_URL", usgs_api.DEFAULT_API_URL)

# ── Storage paths ──
VOLUME_ROOT = Path("/Volumes/weather_api/raw/raw_files")
WEATHER_DIR = VOLUME_ROOT / "weather"
AIR_QUALITY_DIR = VOLUME_ROOT / "air_quality"
EARTHQUAKES_DIR = VOLUME_ROOT / "earthquakes"

# API variable definitions now live in the Python modules (weather_api.py, air_quality_api.py)
# The notebook calls their fetch_* functions instead of reimplementing API requests.

print("✅ Configuration loaded")
print(f"   Weather API:     {WEATHER_API_URL}")
print(f"   Air Quality API: {AIR_QUALITY_API_URL}")
print(f"   USGS API:        {USGS_API_URL}")
print(f"   Storage root:    {VOLUME_ROOT}")

In [0]:
def load_locations():
    """Load and deduplicate locations from config/locations.yaml."""
    with open(LOCATIONS_FILE) as f:
        config = yaml.safe_load(f)
    locations = config["locations"]
    # Deduplicate by (name, latitude, longitude)
    seen = set()
    unique = []
    for loc in locations:
        key = (loc["name"], loc["latitude"], loc["longitude"])
        if key not in seen:
            seen.add(key)
            unique.append(loc)
    if len(unique) < len(locations):
        print(f"⚠️  Removed {len(locations) - len(unique)} duplicate location(s) from config")
    return unique


def sanitize_location_name(name):
    """Convert a location name to a filename-safe string."""
    sanitized = re.sub(r"[^\w\s-]", "", name)
    return re.sub(r"[\s-]+", "_", sanitized).strip("_")


def get_timestamp_utc():
    """Return current UTC timestamp in ISO and filename formats."""
    now = datetime.now(timezone.utc)
    return now.isoformat(), now.strftime("%Y%m%dT%H%M%SZ")


def save_raw_json(payload, filepath):
    """Write JSON atomically — write to temp file, validate, then rename."""
    temp_path = str(filepath) + ".tmp"
    try:
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        # Validate the temp file is parseable JSON before committing
        with open(temp_path, "r", encoding="utf-8") as f:
            json.load(f)
        os.rename(temp_path, str(filepath))
    except Exception:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise


def build_wrapper(source, raw_response, api_endpoint, location_name=None,
                   latitude=None, longitude=None, request_params=None):
    """Wrap the raw API response with ingestion metadata (does not modify the payload)."""
    iso_ts, _ = get_timestamp_utc()
    wrapper = {
        "metadata": {
            "source": source,
            "retrieval_timestamp_utc": iso_ts,
            "api_endpoint": api_endpoint,
        },
        "raw_response": raw_response,
    }
    if location_name is not None:
        wrapper["metadata"]["location_name"] = location_name
    if latitude is not None:
        wrapper["metadata"]["latitude"] = latitude
    if longitude is not None:
        wrapper["metadata"]["longitude"] = longitude
    if request_params is not None:
        wrapper["metadata"]["request_params"] = request_params
    return wrapper

In [0]:
def ingest_weather(locations):
    """Fetch comprehensive weather data for each location and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  WEATHER INGESTION — {len(locations)} locations")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    for loc in locations:
        loc_name = loc["name"]
        try:
            # Fetch via the weather_api module (single source of truth for API variables)
            raw_data, params = weather_api.fetch_weather(
                loc["latitude"], loc["longitude"], api_url=WEATHER_API_URL
            )

            wrapper = build_wrapper(
                source="weather",
                raw_response=raw_data,
                api_endpoint=WEATHER_API_URL,
                location_name=loc_name,
                latitude=loc["latitude"],
                longitude=loc["longitude"],
                request_params=params,
            )

            _, file_ts = get_timestamp_utc()
            safe_name = sanitize_location_name(loc_name)
            filename = f"weather_{safe_name}_{file_ts}.json"
            filepath = WEATHER_DIR / filename
            save_raw_json(wrapper, filepath)

            print(f"✅ weather — {loc_name}: {filepath}")
            results["success"].append({"location": loc_name, "path": str(filepath)})

        except Exception as e:
            print(f"❌ weather — {loc_name}: {e}")
            results["errors"].append({"location": loc_name, "error": str(e)})

    return results

In [0]:
def ingest_air_quality(locations):
    """Fetch comprehensive air quality data for each location and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  AIR QUALITY INGESTION — {len(locations)} locations")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    for loc in locations:
        loc_name = loc["name"]
        try:
            # Fetch via the air_quality_api module (single source of truth for API variables)
            raw_data, params = air_quality_api.fetch_air_quality(
                loc["latitude"], loc["longitude"], api_url=AIR_QUALITY_API_URL
            )

            wrapper = build_wrapper(
                source="air_quality",
                raw_response=raw_data,
                api_endpoint=AIR_QUALITY_API_URL,
                location_name=loc_name,
                latitude=loc["latitude"],
                longitude=loc["longitude"],
                request_params=params,
            )

            _, file_ts = get_timestamp_utc()
            safe_name = sanitize_location_name(loc_name)
            filename = f"air_quality_{safe_name}_{file_ts}.json"
            filepath = AIR_QUALITY_DIR / filename
            save_raw_json(wrapper, filepath)

            print(f"✅ air_quality — {loc_name}: {filepath}")
            results["success"].append({"location": loc_name, "path": str(filepath)})

        except Exception as e:
            print(f"❌ air_quality — {loc_name}: {e}")
            results["errors"].append({"location": loc_name, "error": str(e)})

    return results

In [0]:
def ingest_earthquakes():
    """Fetch the complete USGS earthquake GeoJSON feed and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  EARTHQUAKE INGESTION — USGS all_day feed")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    try:
        # Fetch via the usgs_api module
        raw_data, _ = usgs_api.fetch_earthquakes(api_url=USGS_API_URL)

        feature_count = len(raw_data.get("features", []))

        # Wrap with metadata (no location — global feed)
        wrapper = build_wrapper(
            source="earthquakes",
            raw_response=raw_data,
            api_endpoint=USGS_API_URL,
        )

        _, file_ts = get_timestamp_utc()
        filename = f"earthquakes_{file_ts}.json"
        filepath = EARTHQUAKES_DIR / filename
        save_raw_json(wrapper, filepath)

        print(f"✅ earthquakes — {feature_count} events: {filepath}")
        results["success"].append({"events": feature_count, "path": str(filepath)})

    except Exception as e:
        print(f"❌ earthquakes: {e}")
        results["errors"].append({"error": str(e)})

    return results

In [0]:
# ── Read source parameter (set by job tasks, defaults to "all" for interactive use) ──
SOURCE = dbutils.widgets.get("source").strip().lower()

def main(source="all"):
    """Run ingestion for the specified source(s) and report results."""
    print("=" * 60)
    print("  RAW API DATA INGESTION — Environmental Data Lakehouse")
    print(f"  Run time (UTC): {datetime.now(timezone.utc).isoformat()}")
    print(f"  Source: {source}")
    print("=" * 60)

    # Validate source parameter
    valid_sources = {"weather", "air_quality", "earthquakes", "all"}
    if source not in valid_sources:
        raise ValueError(f"Unknown source: '{source}'. Must be one of: {', '.join(sorted(valid_sources))}")

    # Determine which sources to run
    sources_to_run = ["weather", "air_quality", "earthquakes"] if source == "all" else [source]

    # Verify storage directories exist for the sources being run
    dir_map = {"weather": WEATHER_DIR, "air_quality": AIR_QUALITY_DIR, "earthquakes": EARTHQUAKES_DIR}
    for s in sources_to_run:
        dir_map[s].mkdir(parents=True, exist_ok=True)

    # Verify API URLs are loaded
    url_map = {"weather": WEATHER_API_URL, "air_quality": AIR_QUALITY_API_URL, "earthquakes": USGS_API_URL}
    for s in sources_to_run:
        if not url_map[s]:
            raise RuntimeError(f"API URL for '{s}' is not set — check .env at {PROJECT_ROOT / '.env'}")

    # Load and deduplicate locations (needed for weather and air_quality)
    locations = None
    if source in ("weather", "air_quality", "all"):
        locations = load_locations()
        print(f"\n📡 Loaded {len(locations)} unique locations from config")

    all_results = {}

    if source in ("weather", "all"):
        try:
            all_results["weather"] = ingest_weather(locations)
        except Exception as e:
            print(f"❌ Weather ingestion failed entirely: {e}")
            if source == "weather":
                raise  # Fail the task so dependent Bronze task is skipped
            all_results["weather"] = {"success": [], "errors": [{"error": str(e)}]}

    if source in ("air_quality", "all"):
        time.sleep(1)
        try:
            all_results["air_quality"] = ingest_air_quality(locations)
        except Exception as e:
            print(f"❌ Air quality ingestion failed entirely: {e}")
            if source == "air_quality":
                raise
            all_results["air_quality"] = {"success": [], "errors": [{"error": str(e)}]}

    if source in ("earthquakes", "all"):
        time.sleep(1)
        try:
            all_results["earthquakes"] = ingest_earthquakes()
        except Exception as e:
            print(f"❌ Earthquake ingestion failed entirely: {e}")
            if source == "earthquakes":
                raise
            all_results["earthquakes"] = {"success": [], "errors": [{"error": str(e)}]}

    # Summary
    print(f"\n{'='*60}")
    print("  INGESTION SUMMARY")
    print(f"{'='*60}")
    for src, res in all_results.items():
        s = len(res["success"])
        e = len(res["errors"])
        print(f"  {src:15s} — ✅ {s} succeeded, ❌ {e} failed")

    # For single-source mode: fail the task if any errors occurred
    # so the dependent Bronze task is skipped by the job scheduler
    if source != "all":
        for src, res in all_results.items():
            if res["errors"]:
                error_summary = "; ".join(str(e.get("error", e.get("location", ""))) for e in res["errors"])
                raise RuntimeError(f"{source} ingestion completed with {len(res['errors'])} error(s): {error_summary}")

    return all_results


# Run ingestion for the specified source
results = main(SOURCE)

In [0]:
def validate_ingestion(results):
    """Verify that files saved in the current run are valid JSON with complete API responses."""
    print(f"\n{'='*60}")
    print("  VALIDATION (current run only)")
    print(f"{'='*60}")

    # Collect files from the current run's results
    files_to_validate = []
    for source, res in results.items():
        for item in res["success"]:
            if "path" in item:
                files_to_validate.append((source, Path(item["path"])))

    print(f"\n📋 Validating {len(files_to_validate)} file(s) from current run...")

    valid = 0
    invalid = 0

    for source, f in files_to_validate:
        try:
            with open(f, "r", encoding="utf-8") as fh:
                data = json.load(fh)

            # Check for expected wrapper structure
            if "metadata" not in data or "raw_response" not in data:
                print(f"❌ {f.name}: missing wrapper structure")
                invalid += 1
                continue

            # Check raw_response is non-empty
            if not data["raw_response"]:
                print(f"❌ {f.name}: empty raw_response")
                invalid += 1
                continue

            valid += 1
            print(f"✅ {f.name}")

        except json.JSONDecodeError:
            print(f"❌ {f.name}: invalid JSON")
            invalid += 1
        except Exception as e:
            print(f"❌ {f.name}: {e}")
            invalid += 1

    print(f"\n  ✅ Valid files: {valid}")
    print(f"  ❌ Invalid files: {invalid}")

    # Show sample file contents (metadata + key counts)
    if files_to_validate:
        _, sample = files_to_validate[-1]
        with open(sample, "r", encoding="utf-8") as fh:
            sample_data = json.load(fh)
        print(f"\n📋 Sample file: {sample.name}")
        print(f"   Source: {sample_data['metadata']['source']}")
        print(f"   Retrieved: {sample_data['metadata']['retrieval_timestamp_utc']}")
        raw = sample_data["raw_response"]
        if "current" in raw:
            print(f"   Current fields: {len(raw.get('current', {}))}")
        if "hourly" in raw:
            print(f"   Hourly fields: {len(raw.get('hourly', {}))}")
            print(f"   Hourly records: {len(raw.get('hourly', {}).get('time', []))}")
        if "daily" in raw:
            print(f"   Daily fields: {len(raw.get('daily', {}))}")
        if "features" in raw:
            print(f"   Earthquake features: {len(raw.get('features', []))}")

    print(f"\n{'='*60}")
    print("  VALIDATION COMPLETE")
    print(f"{'='*60}")

    return {"valid": valid, "invalid": invalid, "total_files": len(files_to_validate)}


validation = validate_ingestion(results)